# Week 1 Assignment — Putting It All Together
**Week 1: Python Foundations for Generative AI — "The Interpreter"**

This assignment pulls together everything from Classes 1-3:
- Variables, types, conditionals, loops (Class 1)
- Lists, tuples, dicts, sets (Class 1)
- String methods, f-strings, JSON, file I/O (Class 2)
- Environment variables, `requests`, and calling a real LLM API (Class 3)

Work through **Part A → B → C** in order, then finish with the **Capstone** in Part D, which combines all three classes into one small project.

No solutions are provided — each cell has a `# TODO` marking where your code goes. Read the acceptance criteria before you start each exercise; that's what "done" looks like.

## Setup
Part C and the Capstone need a real LLM call, so they need a `GROQ_API_KEY`. Parts A and B are pure Python — no key needed.

**Running in Google Colab:**
1. Get a free key from https://console.groq.com/keys
2. Click the key icon (🔑 "Secrets") in the left sidebar
3. Add a secret named `GROQ_API_KEY`, paste your key, and toggle **Notebook access** on
4. Run the two setup cells below

In [ ]:
!pip install -q groq requests

In [ ]:
import os
import json

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    print(
        "No API key found — Parts A and B will still work.\n"
        "In Colab: add a secret named GROQ_API_KEY via the 🔑 Secrets panel and enable notebook access.\n"
        "Elsewhere: set GROQ_API_KEY as an environment variable before launching Jupyter."
    )
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("Found GROQ_API_KEY.")

In [ ]:
def call_llm(prompt, system_prompt=None, model="llama-3.3-70b-versatile", max_tokens=300):
    """Send a prompt to Groq and return the text, or an error message. Reused across Part C and the Capstone."""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return "Error: GROQ_API_KEY is not set."
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(model=model, messages=messages, max_tokens=max_tokens)
        return response.choices[0].message.content
    except Exception as e:
        return f"Request failed: {e}"

---
## Part A — Python Foundations (Class 1 Recap)
Variables, types, conditionals, loops, and the four containers.

### A1 — Model Router
Write `route_model(prompt_length, priority)` that returns a model name string using these rules, checked in order:
1. If `priority == "high"` → `"llama-3.3-70b-versatile"`
2. Else if `prompt_length > 500` → `"llama-3.3-70b-versatile"`
3. Otherwise → `"llama-3.1-8b-instant"`

Call it with 4 different `(prompt_length, priority)` pairs, covering all three rule branches at least once.

**Acceptance criteria:** all 4 calls print the correct model name for their inputs.

In [ ]:
# TODO: write route_model(prompt_length, priority) and test it with 4 calls


### A2 — Retry Loop Simulator
A flaky API returns this sequence of status codes on successive attempts: `[500, 429, 200]`. Write a `while` loop that "calls" the API (just reads the next status from the list) up to 3 times, printing each attempt, and **stops as soon as it gets a 200** — don't keep looping after success.

**Acceptance criteria:** prints one line per attempt actually made (not more), and a final line reporting success and how many attempts it took.

In [ ]:
fake_statuses = [500, 429, 200]
# TODO: loop through fake_statuses with a while loop, stop early on 200, report attempts used


### A3 — Token Budget Tracker
Given the list of request dicts below, compute **total tokens per user** using a `dict` (accumulate as you loop), and print the **set of unique usernames**.

```python
requests_log = [
    {"user": "ada", "tokens": 120},
    {"user": "grace", "tokens": 340},
    {"user": "ada", "tokens": 75},
    {"user": "alan", "tokens": 200},
    {"user": "grace", "tokens": 60},
]
```

**Acceptance criteria:** prints a dict of per-user totals (`ada: 195`, `grace: 400`, `alan: 200`) and prints the 3 unique usernames.

In [ ]:
requests_log = [
    {"user": "ada", "tokens": 120},
    {"user": "grace", "tokens": 340},
    {"user": "ada", "tokens": 75},
    {"user": "alan", "tokens": 200},
    {"user": "grace", "tokens": 60},
]
# TODO: build a dict of total tokens per user, and print the set of unique usernames


---
## Part B — Text & JSON (Class 2 Recap)
String methods, f-strings, JSON, and file I/O.

### B1 — Prompt Sanitizer
Given the messy prompts below, clean each one into a single-spaced, lowercase sentence using only string methods (`.strip()`, `.lower()`, `.split()`, `.join()` — no regex needed).

```python
messy_prompts = [
    "  Explain   RECURSION please  ",
    "WHAT is   a Dictionary?",
    "  summarize\tthis   ARTICLE  ",
]
```

**Acceptance criteria:** prints a cleaned list where every entry is lowercase with single spaces and no leading/trailing whitespace.

In [ ]:
messy_prompts = [
    "  Explain   RECURSION please  ",
    "WHAT is   a Dictionary?",
    "  summarize\tthis   ARTICLE  ",
]
# TODO: clean each prompt into a single-spaced, lowercase string


### B2 — Dynamic System Prompt Builder
Write `build_system_prompt(persona, rules)` where `rules` is a list of strings, returning an f-string like:

```
You are {persona}. Follow these rules:
- rule one
- rule two
```

Call it with 2 different personas and rule lists.

**Acceptance criteria:** both calls return a string containing the persona name and every rule, each rule on its own bulleted line.

In [ ]:
# TODO: write build_system_prompt(persona, rules) and call it with 2 different personas


### B3 — Parse & Summarize a Chat Export
Parse the JSON string below (a chat export with a `tokens` count per message), compute **total tokens per role** using a dict, and write a summary report to `chat_summary.json` using `json.dump`.

```python
export = '''
[
  {"role": "user", "content": "hi", "tokens": 5},
  {"role": "assistant", "content": "hello!", "tokens": 8},
  {"role": "user", "content": "explain lists", "tokens": 12},
  {"role": "assistant", "content": "a list is...", "tokens": 40}
]
'''
```

**Acceptance criteria:** `chat_summary.json` exists and contains correct total token counts for `"user"` and `"assistant"`.

In [ ]:
export = '''
[
  {"role": "user", "content": "hi", "tokens": 5},
  {"role": "assistant", "content": "hello!", "tokens": 8},
  {"role": "user", "content": "explain lists", "tokens": 12},
  {"role": "assistant", "content": "a list is...", "tokens": 40}
]
'''
# TODO: json.loads(export), total tokens per role, then json.dump the summary to chat_summary.json


---
## Part C — APIs & Your First AI Calls (Class 3 Recap)
Environment variables, `requests`, and the Groq LLM API.

### C1 — Safe Multi-Key Loader
Write `load_required_env(names)` that takes a list of environment variable names and returns a dict of `{name: value}` **only for the ones that are set**, printing a specific "Missing {name}" warning for each one that isn't — without raising an exception.

Test it with `["GROQ_API_KEY", "SOME_FAKE_KEY_THAT_DOES_NOT_EXIST"]`.

**Acceptance criteria:** returns a dict containing only the present variable(s), and prints a warning naming the specific missing variable.

In [ ]:
# TODO: write load_required_env(names) and test it with GROQ_API_KEY + a fake missing name


### C2 — Public API Health Check
Make GET requests to `https://httpbin.org/status/200` and `https://httpbin.org/status/500` (no key needed). For each, print the status code and a label — `"healthy"` if the status is below 400, otherwise `"unhealthy"`.

**Acceptance criteria:** prints both status codes with the correct healthy/unhealthy label for each.

In [ ]:
import requests
# TODO: GET both URLs, print status code + healthy/unhealthy label for each


### C3 — Multi-Turn Conversation Helper
Write `chat_turn(history, user_message)` that: appends `{"role": "user", "content": user_message}` to `history`, calls `call_llm` with a prompt built from the **full history** (hint: you'll need to pass all prior turns as `messages`, not just the latest prompt — feel free to write a small variant of `call_llm` that accepts a messages list directly), appends the assistant's reply to `history`, and returns the updated `history`.

Call it twice in a row with two different follow-up messages, using the same `history` list each time.

**Acceptance criteria:** after 2 calls, `history` has exactly 4 entries (2 user, 2 assistant), and each assistant entry is non-empty when `GROQ_API_KEY` is set.

In [ ]:
history = []
# TODO: write chat_turn(history, user_message) and call it twice with the same history list


---
## Part D — Capstone: Mini Support Ticket Triager
Combine every concept from this week into one small pipeline: clean raw ticket text (Class 2), ask an LLM to classify and summarize each one (Class 3), tally results with a dict (Class 1), and save a structured JSON report (Class 2).

```python
raw_tickets = [
    "  my APP keeps   crashing every time i open it!! please help ASAP  ",
    "just wanted to say the new dashboard looks great, nice work",
    "I can't log in,   it's been down for   TWO days and I'm losing customers",
    "  minor typo on the pricing page,   'recieve' should be 'receive'  ",
]
```

**Steps:**
1. **Clean** each raw ticket string (Class 2 string methods) into a normalized single-spaced sentence.
2. **Classify** — for each cleaned ticket, build a system prompt (f-string) instructing the model to reply with *only* a compact JSON object like `{"urgency": "low|medium|high", "summary": "..."}`, then call `call_llm(cleaned_ticket, system_prompt=...)`.
3. **Parse safely** — `json.loads` the model's reply inside a `try/except`; if parsing fails, fall back to `{"urgency": "unknown", "summary": cleaned_ticket[:60]}` rather than crashing.
4. **Tally** — count how many tickets fall into each urgency level using a `dict`.
5. **Save** — write a report to `tickets_report.json` containing the full per-ticket results list and the urgency tally.
6. **Report** — print an f-string summary, e.g. `"4 tickets processed — 1 high, 1 medium, 2 low"`.

**Acceptance criteria:** `tickets_report.json` exists with one entry per ticket (each containing the original text, urgency, and summary) plus a tally section, and running the notebook with `GROQ_API_KEY` unset still completes without crashing (steps 1, 3, 4, 5, 6 don't require a live key — only step 2's actual call does).

In [ ]:
raw_tickets = [
    "  my APP keeps   crashing every time i open it!! please help ASAP  ",
    "just wanted to say the new dashboard looks great, nice work",
    "I can't log in,   it's been down for   TWO days and I'm losing customers",
    "  minor typo on the pricing page,   'recieve' should be 'receive'  ",
]

# TODO 1: clean each raw ticket into a normalized string

# TODO 2: for each cleaned ticket, build a system prompt and call call_llm(...)

# TODO 3: json.loads the reply inside try/except, falling back to {"urgency": "unknown", "summary": ...}

# TODO 4: tally ticket counts per urgency level in a dict

# TODO 5: write {"tickets": [...], "tally": {...}} to tickets_report.json

# TODO 6: print an f-string summary of the results


### Week 1, closed
If every part above runs end to end, you've used variables, types, conditionals, loops, lists, dicts, sets, string methods, f-strings, JSON parsing and writing, file I/O, environment variables, and a real LLM API call — all in one notebook. That's everything Week 1 covered, working together.